In [ ]:
%load_ext autoreload

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
from collections import defaultdict
import itertools
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm

In [ ]:
%autoreload 2
from src.data import get_electrode_df, add_metadata_features
from src.data_cleaning import prepare_AB_results
from src.models.decoding import run_decoding_population, run_decoding_model_comparison_population

In [ ]:
epochs_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))

electrodes_paths = list(Path("outputs/causal4/find_speech_responsive").glob("*_results.csv"))

A_result_path = Path("outputs/causal4/unify_As/results.csv")
A_decoders_path = Path("outputs/causal4/unify_As/unified_decoders.pt")

B_annotated_path = Path("outputs/causal4/annotated_B_results.csv")

outdir = "outputs/causal4/behavior_decoding"

In [ ]:
epochs = {re.findall(r"(EC[\d]+)_epo", str(path))[0]: mne.read_epochs(path, verbose=False)
          for path in epochs_paths}
for ep in epochs.values():
    ep.metadata = add_metadata_features(ep.metadata)

In [ ]:
A_decoders = torch.load(A_decoders_path)

In [ ]:
A_results, B_results = prepare_AB_results(A_result_path, B_annotated_path)

## Decode from manual-labeled electrodes

In [ ]:
manual_elec_labelings = {
    "stack": [
        ("EC243", 102, "bm"),
        ("EC260", 219, "bm"),
        ("EC243", 103, "bm"),
        ("EC243", 103, "pb"),
        ("EC260", 222, "pb"),
        ("EC243", 105, "bm"),
        ("EC260", 220, "pb"),
        ("EC279", 6, "dn"),
        ("EC248", 365, "dn"),
        ("EC253", 196, "dn"),
        ("EC253", 2, "dn"),
        ("EC279", 167, "dn"),
        ("EC287", 59, "dn"),
        ("EC279", 4, "bm"),
        ("EC279", 4, "dn"),
        ("EC278", 27, "bm"),
        ("EC260", 206, "bm"),
        ("EC260", 206, "pb"),
        ("EC278", 90, "dn"),
    ],

    "alligator": [
        ("EC243", 102, "dn"),
        ("EC243", 102, "pb"),
        ("EC260", 204, "dn"),
        ("EC260", 91, "dn"),
        ("EC260", 93, "dn"),
        ("EC243", 197, "bm"),
        ("EC260", 109, "dn"),
        ("EC243", 103, "dn"),
        ("EC278", 121, "bm"),
        ("EC260", 92, "dn"),
        ("EC250", 216, "pb"),
        ("EC243", 213, "dn"),
        ("EC248", 364, "dn"),
        ("EC250", 207, "dn"),
        ("EC248", 253, "dn"),
        ("EC282", 97, "dn"),
        ("EC278", 27, "dn"),
        ("EC282", 115, "pb"),
        ("EC278", 90, "pb"),
        ("EC279", 76, "bm"),
    ],

    "loo": [
        ('EC260', 204, 'dn'),
        ('EC260', 204, 'pb'),
        ('EC260', 91, 'bm'),
        ('EC260', 93, 'pb'),
        ('EC243', 197, 'dn'),
        ('EC260', 219, 'bm'),
        ('EC260', 219, 'dn'),
        ('EC278', 121, 'dn'),
        ('EC243', 119, 'bm'),
        ('EC243', 119, 'dn'),
        ('EC260', 222, 'bm'),
        ('EC260', 222, 'dn'),
        ('EC243', 105, 'pb'),
        ('EC260', 92, 'bm'),
        ('EC250', 216, 'bm'),
        ('EC260', 221, 'dn'),
        ('EC260', 221, 'pb'),
        ('EC248', 381, 'dn'),
        ('EC260', 220, 'dn'),
        ('EC253', 212, 'dn'),
        ('EC250', 215, 'bm'),
        ('EC250', 215, 'dn'),
        ('EC248', 364, 'bm'),
        ('EC260', 76, 'bm'),
        ('EC260', 76, 'dn'),
        ('EC270', 122, 'dn'),
        ('EC282', 116, 'bm'),
        ('EC287', 124, 'pb'),
        ('EC253', 196, 'pb'),
        ('EC248', 253, 'dn'),
        ('EC279', 167, 'pb'),
        ('EC279', 152, 'dn'),
        ('EC279', 152, 'pb'),
        ('EC287', 5, 'pb'),
        ('EC270', 140, 'dn'),
        ('EC260', 206, 'dn'),
        ('EC278', 90, 'bm'),
        ('EC243', 228, 'dn'),
        ('EC243', 72, 'bm'),
        ('EC243', 72, 'dn'),
        ('EC279', 11, 'pb'),
        ('EC279', 76, 'dn'),
    ]
}

In [ ]:
manual_label_df = pd.concat({label: pd.DataFrame(label_df, columns=["subject", "electrode_idx", "phoneme_pair"])
           for label, label_df in manual_elec_labelings.items()}, names=["manual_label"]).droplevel(-1).reset_index()

In [ ]:
len(epochs[subject].info["chs"])

## Decode from stimulus resampled

## Decode from manually labeled electrodes

In [ ]:
manual_decoding_results = {}
for (subject, phoneme_pair, manual_label), rows in tqdm(manual_label_df.groupby(["subject", "phoneme_pair", "manual_label"])):
    elec_idxs = rows.electrode_idx
    assert elec_idxs.nunique() == len(elec_idxs), "Duplicate electrode indices in manual labeling."
    elec_idxs = elec_idxs[elec_idxs < len(epochs[subject].info["chs"])]
    if elec_idxs.empty:
        continue
    elec_idxs = elec_idxs.tolist()

    manual_decoding_results[subject, phoneme_pair, manual_label] = run_decoding_model_comparison_population(
        epochs[subject],
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name=manual_label,
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components=0.95,
        strategy="train-test",
    )

## Decode from A-populations

In [ ]:
A_decoding_results = {}
for pop in tqdm(A_results.itertuples(), total=len(A_results)):
    elec_idxs = A_decoders["populations"][pop.subject, pop.population_name, pop.phoneme_pair]
    A_decoding_results[pop.subject, pop.population_name, pop.phoneme_pair] = run_decoding_model_comparison_population(
        epochs[pop.subject],
        elec_idxs,
        phoneme_pair=pop.phoneme_pair,
        subject=pop.subject,
        population_name=pop.population_name,
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components=0.95,
        strategy="train-test",
    )

In [ ]:
# for pop in A_results.itertuples():
#     elec_idxs = A_decoders["populations"][pop.subject, pop.population_name, pop.phoneme_pair]
#     train_scores, test_scores, outcomes, models = run_decoding_population(
#         epochs[pop.subject],
#         elec_idxs,
#         phoneme_pair=pop.phoneme_pair,
#         subject=pop.subject,
#         population_name=pop.population_name,
#         stride=10,
#         window_size=30,
#         pca_num_components=0.95,
#         strategy="train-test",
#     )
#     break

## Decode from B-populations

In [ ]:
B_decoding_results = {}
for (subject, population_name, phoneme_pair), rows in tqdm(B_results.groupby(["subject", "population_name_fixed", "phoneme_pair"])):
    elec_idxs = rows.electrode_idx
    assert elec_idxs.nunique() == len(elec_idxs)

    B_decoding_results[subject, population_name, phoneme_pair] = run_decoding_model_comparison_population(
        epochs[subject],
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name=population_name,
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components=0.95,
        strategy="train-test",
    )

## Save

In [ ]:
torch.save({"A_decoding_results": A_decoding_results,
            "B_decoding_results": B_decoding_results,
            "manual_decoding_results": manual_decoding_results,
            },
            "20250829_behavior_decoding.pt")

## Summarize

In [ ]:
manual_results_df = pd.concat(manual_decoding_results, names=["subject", "phoneme_pair", "manual_label"], ignore_index=True)

In [ ]:
manual_results_df["diff"] = manual_results_df["full_roc_auc"] - manual_results_df["baseline_roc_auc"]

In [ ]:
# loaded = torch.load("20250829_behavior_decoding.pt")
# A_decoding_results = loaded["A_decoding_results"]
# B_decoding_results = loaded["B_decoding_results"]

In [ ]:
A_results_df = pd.concat(A_decoding_results, names=["subject", "population_name", "phoneme_pair"], ignore_index=True)
B_results_df = pd.concat(B_decoding_results, names=["subject", "population_name", "phoneme_pair"], ignore_index=True)

In [ ]:
A_results_df["diff"] = A_results_df["full_roc_auc"] - A_results_df["baseline_roc_auc"]
B_results_df["diff"] = B_results_df["full_roc_auc"] - B_results_df["baseline_roc_auc"]

In [ ]:
manual_summary = manual_results_df.groupby(["subject", "population", "phoneme_pair", "smin", "smax"])[["baseline_roc_auc", "full_roc_auc", "diff"]].mean()
manual_max_points = manual_summary.groupby(["subject", "population", "phoneme_pair"])["diff"].idxmax()
manual_final_summary = manual_summary.loc[manual_max_points]
# manual_final_summary.sort_values("diff")

In [ ]:
A_summary = A_results_df.groupby(["subject", "population", "phoneme_pair", "smin", "smax"]) \
    [["baseline_roc_auc", "full_roc_auc", "diff"]].mean()
A_max_points = A_summary \
    .groupby(["subject", "population", "phoneme_pair"])["diff"].idxmax()
A_final_summary = A_summary.loc[A_max_points]
# A_final_summary.sort_values("diff")

In [ ]:
B_summary = B_results_df.groupby(["subject", "population", "phoneme_pair", "smin", "smax"])[["baseline_roc_auc", "full_roc_auc", "diff"]].mean()
B_max_points = B_summary.groupby(["subject", "population", "phoneme_pair"])["diff"].idxmax()
B_final_summary = B_summary.loc[B_max_points]
# B_final_summary.sort_values("diff")

In [ ]:
all_summary = pd.concat({"A": A_final_summary, "B": B_final_summary, "manual": manual_final_summary},
                        names=["source"]).reset_index()

In [ ]:
g = sns.displot(all_summary, x="diff", row="source", col="phoneme_pair", height=3, aspect=2)
for ax in g.axes.flat:
    ax.axvline(0, color="red", linestyle="--")